# Simulating control systems with Julia

Control System Engineering (EEE2510). Run each cell with Shift + Enter. Change the numbers and run the cell again to see what happens.

## Set up Julia (once)

In [ ]:
using Pkg
Pkg.add(["ControlSystems", "Plots", "IJulia"])   # run once

using Plots
plotly()      # optional: zoomable plots in Jupyter, like the ones on this page

## 1 · First-order step response

A first-order system $G(s) = \dfrac{K}{\tau s + 1}$ models an RC circuit, a thermometer or a small tank. Its step response is $y(t) = K\left(1 - e^{-t/\tau}\right)$: it reaches 63.2 % of the final value $K$ at $t = \tau$ and settles (within 2 %) after about $4\tau$.

The code builds the transfer function with tf, simulates it with step and plots the result.

In [ ]:
using ControlSystems, Plots

K = 2.0        # steady-state gain
τ = 1.5        # time constant (s)   type \tau then press Tab

G = tf(K, [τ, 1])              # G(s) = K / (τs + 1)
t = 0:0.02:10                  # time from 0 to 10 s
res = step(G, t)               # unit-step response
y = res.y[1, :]

println("Final value   = ", K)
println("Rise time     ≈ ", round(2.2τ, digits = 2), " s")
println("Settling (2%) ≈ ", round(4τ, digits = 2), " s")

plot(t, y, lw = 2, label = "y(t)", xlabel = "t (s)", ylabel = "y",
     title = "First-order step response")
hline!([0.632K], ls = :dash, label = "63.2 % of K")
vline!([τ], ls = :dot, label = "t = τ")

## 2 · Second-order response and specifications

The standard second-order system is $G(s) = \dfrac{\omega_n^2}{s^2 + 2\zeta\omega_n s + \omega_n^2}$. For $0 < \zeta < 1$ (underdamped) the time-domain specifications are

$$M_p = e^{-\zeta\pi/\sqrt{1-\zeta^2}}, \qquad t_p = \frac{\pi}{\omega_d}, \qquad t_s \approx \frac{4}{\zeta\omega_n}, \qquad \omega_d = \omega_n\sqrt{1-\zeta^2}$$

The code computes them with these formulas and compares the peak with the simulated one.

In [ ]:
using ControlSystems, Plots

ωn = 2.0       # natural frequency (rad/s)   type \omega Tab, then n
ζ  = 0.4       # damping ratio               type \zeta Tab

G = tf(ωn^2, [1, 2ζ*ωn, ωn^2])     # standard second-order system
t = 0:0.01:10
y = step(G, t).y[1, :]

if 0 < ζ < 1                       # formulas hold for an underdamped system
    ωd = ωn * sqrt(1 - ζ^2)
    println("Peak overshoot Mp = ", round(100 * exp(-ζ*π/sqrt(1 - ζ^2)), digits = 1), " %")
    println("Peak time      tp = ", round(π/ωd, digits = 2), " s")
    println("Settling time  ts ≈ ", round(4/(ζ*ωn), digits = 2), " s  (2 % band)")
end
println("Measured peak     = ", round(maximum(y), digits = 3))

plot(t, y, lw = 2, label = "ζ = $ζ", xlabel = "t (s)", ylabel = "y",
     title = "Second-order step response")
hline!([0.98, 1.02], ls = :dash, color = :gray, label = "±2 % band")

## 3 · PID control of a third-order plant

The plant $G(s) = \dfrac{1}{(s+1)^3}$ is controlled by a PID controller $C(s) = K_p + \dfrac{K_i}{s} + K_d s = \dfrac{K_d s^2 + K_p s + K_i}{s}$ in a unity-feedback loop. feedback(C * G) gives the closed loop $\dfrac{CG}{1 + CG}$.

The dashed curve is P-only control with the same $K_p$, so you can see what the integral and derivative terms add.

In [ ]:
using ControlSystems, Plots

G = tf(1, [1, 3, 3, 1])        # plant: G(s) = 1/(s + 1)^3

Kp = 2.0                       # proportional gain
Ki = 0.5                       # integral gain
Kd = 1.0                       # derivative gain

C = tf([Kd, Kp, Ki], [1, 0])   # PID: C(s) = Kp + Ki/s + Kd·s
T = feedback(C * G)            # closed loop: CG / (1 + CG)
t = 0:0.05:30
y  = step(T, t).y[1, :]
yP = step(feedback(Kp * G), t).y[1, :]    # P only, for comparison

println("Peak value         = ", round(maximum(y), digits = 3))
println("Steady-state error = ", round(1 - y[end], digits = 3))
println("Stable? ", all(real.(poles(T)) .< 0))

plot(t, y, lw = 2, label = "PID", xlabel = "t (s)", ylabel = "y",
     title = "PID control of 1/(s+1)³", ylims = (-0.5, 2.5))
plot!(t, yP, ls = :dash, label = "P only (Kp = $Kp)")
hline!([1], color = :gray, ls = :dot, label = "set-point")

## 4 · Root locus

For the loop $KG(s)H(s) = \dfrac{K}{s(s+2)(s+4)}$ from the Lesson 12 exercises, the root locus shows where the closed-loop poles go as $K$ grows. The pink dots are the poles for the gain you choose, and the right-hand plot is the closed-loop step response for that gain.

From the Routh array, the locus crosses the $j\omega$ axis at $K = 48$, $\omega = \sqrt{8} = 2.83$ rad/s.

In [ ]:
using ControlSystems, Plots

G = tf(1, [1, 6, 8, 0])          # G(s)H(s) = 1 / (s(s + 2)(s + 4))
K = 10.0                         # controller gain

cl = poles(feedback(K * G))      # closed-loop poles for this K
println("Closed-loop poles: ", round.(cl, digits = 3))
println(all(real.(cl) .< 0) ? "Stable" : "Unstable")

p1 = rlocusplot(G)               # the whole root locus, K from 0 to ∞
scatter!(p1, real.(cl), imag.(cl), ms = 8, label = "poles at K = $K")

t = 0:0.05:20
p2 = plot(t, step(feedback(K * G), t).y[1, :], lw = 2, label = "y(t)",
          xlabel = "t (s)", title = "Closed-loop step response")
plot(p1, p2, layout = (1, 2), size = (900, 400))

## 5 · Bode plot, gain margin and phase margin

For the open loop $G(s) = \dfrac{K}{s(s+1)(s+5)}$, margin finds the gain margin (how much the gain can rise before instability, read at the phase crossover where the phase is $-180°$) and the phase margin (read at the gain crossover where $|G| = 0$ dB). marginplot draws the Bode plot with both margins marked.

In [ ]:
using ControlSystems, Plots

K = 10.0
G = tf(K, [1, 6, 5, 0])          # G(s) = K / (s(s + 1)(s + 5))

wgm, gm, wpm, pm = margin(G)     # crossover frequencies and margins
println("Gain margin  = ", round(20log10(gm[1]), digits = 2), " dB  at ω = ", round(wgm[1], digits = 3), " rad/s")
println("Phase margin = ", round(pm[1], digits = 2), "°   at ω = ", round(wpm[1], digits = 3), " rad/s")

w = exp10.(range(-2, 2, length = 500))   # frequencies from 0.01 to 100 rad/s
marginplot(G, w)                         # Bode plot with the margins marked

## 6 · Tank level control with a PI controller (nonlinear)

The tank from Lesson 1 obeys $A\,\dfrac{dh}{dt} = q_{in} - k\sqrt{h}$, which is nonlinear, so instead of a transfer function we write the simulation loop ourselves with Euler steps. A PI controller sets the inflow, the valve can only open between 0 and 100 %, and at $t = 100$ s the outlet opens 30 % wider (a disturbance).

This example uses plain Julia and Plots only: it shows how to code any model you can write as a differential equation.

In [ ]:
using Plots

r  = 2.0             # level set-point (m)
Kp = 1.5             # proportional gain
Ki = 0.1             # integral gain

# Tank:  A dh/dt = q_in - k*sqrt(h)
function simulate_tank(r, Kp, Ki; A = 2.0, k = 0.4, q_max = 1.0, dt = 0.05, t_end = 200.0)
    t = 0:dt:t_end
    h = zeros(length(t));  h[1] = 0.5     # start at 0.5 m
    q = zeros(length(t))
    integral = 0.0
    for i in 1:length(t)-1
        kout = t[i] < 100 ? k : 1.3k      # at t = 100 s the outlet opens 30 % more
        e = r - h[i]                      # error
        u = Kp*e + Ki*integral            # PI controller
        q[i] = clamp(u, 0, q_max)         # the valve only opens 0 to 100 %
        if u == q[i]                      # anti-windup: integrate only when not saturated
            integral += e*dt
        end
        h[i+1] = max(h[i] + dt*(q[i] - kout*sqrt(h[i]))/A, 0.0)   # Euler step
    end
    q[end] = q[end-1]
    return t, h, q
end

t, h, q = simulate_tank(r, Kp, Ki)
println("Level at t = 99 s : ", round(h[findfirst(≥(99), t)], digits = 3), " m")
println("Final level       : ", round(h[end], digits = 3), " m")

p1 = plot(t, h, lw = 2, label = "level h", ylabel = "h (m)", title = "Tank level with PI control")
hline!(p1, [r], ls = :dash, label = "set-point")
vline!(p1, [100], ls = :dot, color = :gray, label = "disturbance")
p2 = plot(t, q, lw = 2, color = :orange, label = "inflow q_in", xlabel = "t (s)", ylabel = "q (m³/s)")
plot(p1, p2, layout = (2, 1), size = (750, 550))